<a href="https://colab.research.google.com/github/lricci03/Hands-on-ML/blob/main/c12/c12_ex8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 12 - ex8. Build your own CNN from scratch
Try to achieve the highest possible accuracy on MNIST.

## Imports and settings

In [1]:
import torch
import torch.nn as nn
from functools import partial

In [2]:
!pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 23.8 MB/s eta 0:00:00


In [3]:
import torchmetrics

In [4]:
if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

## Basic CNN to tackle MNIST (p. 434)

In [5]:
from functools import partial

In [6]:
DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding='same') # DefaultConv2d acts as Conv2d with different default arguments

In [8]:
model = nn.Sequential(
    DefaultConv2d(in_channels=1, out_channels=64, kernel_size = 7), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(in_channels=64, out_channels=128), nn.ReLU(),
    DefaultConv2d(in_channels=128, out_channels=128), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(in_channels=128, out_channels=256), nn.ReLU(),
    DefaultConv2d(in_channels=256, out_channels=256), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(in_features=2304, out_features=128), nn.ReLU(), # the pool layers reduce the dimensional size to 3x3, 256 channels: 9x256=2304
    nn.Dropout(0.5),
    nn.Linear(in_features=128, out_features=64), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features=64, out_features=10),
).to(device)

## Import MNIST

In [10]:
import torchvision
import torchvision.transforms.v2 as T

In [11]:
toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

In [13]:
train_and_valid_data = torchvision.datasets.MNIST(
    root='datasets', train=True, download=True, transform=toTensor)
test_data = torchvision.datasets.MNIST(
    root='datasets', train=False, download=True, transform=toTensor)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 481kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.50MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.66MB/s]


MNIST contains 10'000 test instances and 60'000 train instances

In [17]:
torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(train_and_valid_data, [55_000, 5_000])

## Create data loaders

In [18]:
from torch.utils.data import DataLoader

In [20]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

## Train function

In [21]:
import copy
import time

In [22]:
def train(model, optimizer, criterion, train_loader, valid_loader, n_epochs):
  train_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
  valid_accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=10).to(device)

  best_acc = 0.0
  best_state = None

  for epoch in range(n_epochs):
    model.train()
    train_accuracy.reset()
    total_loss = 0.
    t0 = time.time()

    for X_train_batch, y_train_batch in train_loader:
      X_train_batch, y_train_batch = X_train_batch.to(device), y_train_batch.to(device)
      y_pred = model(X_train_batch)
      loss = criterion(y_pred, y_train_batch)
      total_loss += loss.item()
      train_accuracy.update(y_pred, y_train_batch)
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    epoch_accuracy = train_accuracy.compute()
    mean_loss = total_loss/len(train_loader)

    with torch.no_grad():
      model.eval()
      valid_accuracy.reset()

      for X_valid_batch, y_valid_batch in valid_loader:
        X_valid_batch, y_valid_batch = X_valid_batch.to(device), y_valid_batch.to(device)
        y_valid_pred = model(X_valid_batch)
        valid_accuracy.update(y_valid_pred, y_valid_batch)
      epoch_valid_accuracy = valid_accuracy.compute()

      # we save the best model. We don't stop if validation decresaes bc w/ mini-batch sometimes accuracy decreases but then increases again.
      if epoch_valid_accuracy > best_acc:
        best_acc = epoch_valid_accuracy
        best_state = copy.deepcopy(model.state_dict())

    t1 = time.time()

    print(f'Epoch {epoch + 1}/{n_epochs}: Train Loss: {mean_loss:.4f} Train Accuracy: {epoch_accuracy:.4f}, Validation Accuracy: {epoch_valid_accuracy:.4f}'
          f'in {t1-t0:.1f}s')
  if best_state is not None:
    model.load_state_dict(best_state)

  return model, best_acc


In [24]:
learning_rate = 0.001
betas = (0.9, 0.999)
optimizer = torch.optim.NAdam(model.parameters(), betas = betas, lr = learning_rate)
xentropy = nn.CrossEntropyLoss()

In [25]:
best_model, best_acc = train(model, optimizer, xentropy, train_loader, valid_loader, 10)

Epoch 1/10: Train Loss: 0.4272 Train Accuracy: 0.8640, Validation Accuracy: 0.9812in 21.1s
Epoch 2/10: Train Loss: 0.1155 Train Accuracy: 0.9728, Validation Accuracy: 0.9802in 19.8s
Epoch 3/10: Train Loss: 0.0860 Train Accuracy: 0.9793, Validation Accuracy: 0.9860in 20.4s
Epoch 4/10: Train Loss: 0.0692 Train Accuracy: 0.9841, Validation Accuracy: 0.9904in 19.8s
Epoch 5/10: Train Loss: 0.0570 Train Accuracy: 0.9863, Validation Accuracy: 0.9904in 20.3s
Epoch 6/10: Train Loss: 0.0545 Train Accuracy: 0.9874, Validation Accuracy: 0.9858in 20.3s
Epoch 7/10: Train Loss: 0.0504 Train Accuracy: 0.9892, Validation Accuracy: 0.9888in 19.6s
Epoch 8/10: Train Loss: 0.0427 Train Accuracy: 0.9901, Validation Accuracy: 0.9904in 20.4s
Epoch 9/10: Train Loss: 0.0401 Train Accuracy: 0.9904, Validation Accuracy: 0.9890in 19.7s
Epoch 10/10: Train Loss: 0.0377 Train Accuracy: 0.9921, Validation Accuracy: 0.9886in 20.1s


## Inception modules

The following inception module has 64 + 128 + 48 + 48 = 288 output channels

In [27]:
from torch.nn.modules.pooling import MaxPool2d

class InceptionModule(nn.Module):
  def __init__(self, n_inputs):
    super().__init__()
    self.conv0 = nn.Sequential(
        nn.Conv2d(in_channels=n_inputs, out_channels=64, kernel_size=1, padding='same'), nn.ReLU())
    self.conv1 = nn.Sequential(
        nn.Conv2d(in_channels=n_inputs, out_channels=64, kernel_size=1, padding='same'), nn.ReLU(),
        nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding='same'), nn.ReLU())
    self.conv2 = nn.Sequential(
        nn.Conv2d(in_channels=n_inputs, out_channels=16, kernel_size=1, padding='same'), nn.ReLU(),
        nn.Conv2d(in_channels=16, out_channels=48, kernel_size=5, padding='same'), nn.ReLU())
    self.conv3 = nn.Sequential(
        nn.MaxPool2d(kernel_size=3, stride=1, padding=1), # stride=1, padding=1 (this depends on kernel_size=3) so the in & out have same dim
        nn.Conv2d(in_channels=n_inputs, out_channels=48, kernel_size=1, padding='same'), nn.ReLU())

  def forward(self,X):
    return torch.cat([self.conv0(X), self.conv1(X), self.conv2(X), self.conv3(X)], dim=1)


Create a simple model with:
- conv layer, maxpool
- inception module, maxpool
- two fully connected layers with dropout
- output layer

In [32]:
model2 = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=64, kernel_size=7, padding='same'),
    nn.ReLU(),
    InceptionModule(n_inputs=64),
    nn.MaxPool2d(kernel_size=2), # this divides the height and width by 2
    nn.Flatten(),
    nn.Linear(in_features = 56448, out_features = 144), # 288 out channels, 14x14 spacial dimension. Thus
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features = 144, out_features = 77),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features = 77, out_features = 10)
).to(device)

Train model2

In [33]:
learning_rate = 0.001
betas = (0.9, 0.999)
optimizer = torch.optim.NAdam(model2.parameters(), betas = betas, lr = learning_rate)
xentropy = nn.CrossEntropyLoss()

In [34]:
best_model, best_acc = train(model2, optimizer, xentropy, train_loader, valid_loader, 10)

Epoch 1/10: Train Loss: 0.3491 Train Accuracy: 0.8918, Validation Accuracy: 0.9806in 24.6s
Epoch 2/10: Train Loss: 0.1433 Train Accuracy: 0.9590, Validation Accuracy: 0.9846in 24.4s
Epoch 3/10: Train Loss: 0.1111 Train Accuracy: 0.9680, Validation Accuracy: 0.9876in 24.4s
Epoch 4/10: Train Loss: 0.0933 Train Accuracy: 0.9736, Validation Accuracy: 0.9884in 24.7s
Epoch 5/10: Train Loss: 0.0846 Train Accuracy: 0.9750, Validation Accuracy: 0.9898in 24.3s
Epoch 6/10: Train Loss: 0.0736 Train Accuracy: 0.9785, Validation Accuracy: 0.9910in 24.2s
Epoch 7/10: Train Loss: 0.0669 Train Accuracy: 0.9800, Validation Accuracy: 0.9894in 24.1s
Epoch 8/10: Train Loss: 0.0619 Train Accuracy: 0.9817, Validation Accuracy: 0.9918in 24.3s
Epoch 9/10: Train Loss: 0.0604 Train Accuracy: 0.9824, Validation Accuracy: 0.9900in 24.0s
Epoch 10/10: Train Loss: 0.0545 Train Accuracy: 0.9834, Validation Accuracy: 0.9918in 23.9s


## Residual Units

In [37]:
import torch.nn.functional as F

class ResidUnit(nn.Module):
  def __init__(self, n_inputs, n_outputs, stride=1):
    super().__init__()
    self.body = nn.Sequential(
        nn.Conv2d(in_channels=n_inputs, out_channels=n_outputs, kernel_size=3, stride=stride, padding=1), # padding=1 preserves spacial dim when stride=1
        nn.BatchNorm2d(n_outputs),
        nn.ReLU(),
        nn.Conv2d(in_channels=n_outputs, out_channels=n_outputs, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(n_outputs)
    )
    if stride==1:
      self.skip = nn.Identity()
    else:
      self.skip = nn.Sequential(
          nn.Conv2d(in_channels=n_inputs, out_channels=n_outputs, kernel_size=1, stride=stride),
          nn.BatchNorm2d(n_outputs))

  def forward(self,X):
    return F.relu(self.body(X) + self.skip(X))

Create a simple model with:
- conv layer, maxpool
- 2 x residual unit with n_outputs = n_inputs
- 2 x residual unit with n_outputs = 2*n_inputs
- Global average pool
- fully connected linear output layer

In [46]:
model3 = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=64, kernel_size=7, padding='same'),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    ResidUnit(n_inputs=64, n_outputs=64, stride=1),
    ResidUnit(n_inputs=64, n_outputs=64, stride=1),
    ResidUnit(n_inputs=64, n_outputs=128, stride=2),
    ResidUnit(n_inputs=128, n_outputs=128, stride=1),
    nn.AdaptiveAvgPool2d(output_size=1), # or nn.AvgPool2d(kernel_size=14) bc we divide spatial dim by 2 with MaxPool2d
    nn.Flatten(),
    nn.Linear(in_features=128, out_features=10)
).to(device)

Train model3

In [47]:
learning_rate = 0.001
betas = (0.9, 0.999)
optimizer = torch.optim.NAdam(model3.parameters(), betas = betas, lr = learning_rate)
xentropy = nn.CrossEntropyLoss()

In [48]:
best_model, best_acc = train(model3, optimizer, xentropy, train_loader, valid_loader, 10)

Epoch 1/10: Train Loss: 0.1114 Train Accuracy: 0.9675, Validation Accuracy: 0.9896in 24.1s
Epoch 2/10: Train Loss: 0.0450 Train Accuracy: 0.9860, Validation Accuracy: 0.9898in 24.9s
Epoch 3/10: Train Loss: 0.0362 Train Accuracy: 0.9890, Validation Accuracy: 0.9872in 24.4s
Epoch 4/10: Train Loss: 0.0296 Train Accuracy: 0.9911, Validation Accuracy: 0.9896in 24.3s
Epoch 5/10: Train Loss: 0.0259 Train Accuracy: 0.9921, Validation Accuracy: 0.9902in 24.0s
Epoch 6/10: Train Loss: 0.0242 Train Accuracy: 0.9920, Validation Accuracy: 0.9916in 23.8s
Epoch 7/10: Train Loss: 0.0197 Train Accuracy: 0.9937, Validation Accuracy: 0.9934in 24.2s
Epoch 8/10: Train Loss: 0.0174 Train Accuracy: 0.9946, Validation Accuracy: 0.9930in 24.1s
Epoch 9/10: Train Loss: 0.0171 Train Accuracy: 0.9944, Validation Accuracy: 0.9944in 24.5s
Epoch 10/10: Train Loss: 0.0139 Train Accuracy: 0.9957, Validation Accuracy: 0.9956in 24.1s


## Squeeze and Excitation SE Block

In [59]:
class SEBlock(nn.Module):
  def __init__(self, n_inputs):
    super().__init__()
    self.n_inputs = n_inputs
    self.squeeze = nn.AdaptiveAvgPool2d(output_size=1)
    bottleneck_channels = max(1, n_inputs // 16)
    self.excitation = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=n_inputs, out_features=bottleneck_channels),
        nn.ReLU(),
        nn.Linear(in_features=bottleneck_channels, out_features=n_inputs),
        nn.Sigmoid()
    )

  def forward(self,X):
    scale = self.excitation(self.squeeze(X)).reshape(-1,self.n_inputs,1,1) # scale has shape: size of batch X * n_inputs * 1 * 1
    return scale * X  # X has shape: size of batch X * n_inputs * width * height

### SE-ResNet unit (SE Block w/ Residual Unit)

To define a SE-ResNet unit we can't simply combine a residual unit and an SE block, because we **first** have to apply the SE block to the residual unit body and **then** add the skip connection.

The ouput of the *body* of the residual unit is a layer with n_outputs channels.

In [55]:
Mod = ResidUnit(64,64,1).body

In [56]:
Mod

Sequential(
  (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)

In [60]:
class SE_ResNet(nn.Module):
  def __init__(self, n_inputs, n_outputs, stride=1):
    super().__init__()
    res_unit = ResidUnit(n_inputs, n_outputs, stride)
    self.resnet_body = res_unit.body
    self.resnet_skip = res_unit.skip
    self.seblock = SEBlock(n_outputs)

  def forward(self,X):
    res_unit = self.resnet_body(X)
    return F.relu(self.seblock(res_unit) + self.resnet_skip(X))

Create a simple model with:
- conv layer, maxpool
- 2 x SE-ResNet unit with n_outputs = n_inputs
- 2 x SE-ResNet unit with n_outputs = 2*n_inputs
- Global average pool
- fully connected linear output layer

This is model3 with modifications the SE-modification: SE_ResNet instead of ResidUnit.

In [61]:
model4 = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=64, kernel_size=7, padding='same'),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    SE_ResNet(n_inputs=64, n_outputs=64, stride=1),
    SE_ResNet(n_inputs=64, n_outputs=64, stride=1),
    SE_ResNet(n_inputs=64, n_outputs=128, stride=2),
    SE_ResNet(n_inputs=128, n_outputs=128, stride=1),
    nn.AdaptiveAvgPool2d(output_size=1), # or nn.AvgPool2d(kernel_size=14) bc we divide spatial dim by 2 with MaxPool2d
    nn.Flatten(),
    nn.Linear(in_features=128, out_features=10)
).to(device)

Train model4

In [62]:
learning_rate = 0.001
betas = (0.9, 0.999)
optimizer = torch.optim.NAdam(model4.parameters(), betas = betas, lr = learning_rate)
xentropy = nn.CrossEntropyLoss()

In [63]:
best_model, best_acc = train(model4, optimizer, xentropy, train_loader, valid_loader, 10)

Epoch 1/10: Train Loss: 0.1229 Train Accuracy: 0.9654, Validation Accuracy: 0.9764in 31.3s
Epoch 2/10: Train Loss: 0.0457 Train Accuracy: 0.9857, Validation Accuracy: 0.9834in 31.6s
Epoch 3/10: Train Loss: 0.0372 Train Accuracy: 0.9881, Validation Accuracy: 0.9870in 33.7s
Epoch 4/10: Train Loss: 0.0302 Train Accuracy: 0.9909, Validation Accuracy: 0.9872in 36.8s
Epoch 5/10: Train Loss: 0.0269 Train Accuracy: 0.9916, Validation Accuracy: 0.9926in 31.5s
Epoch 6/10: Train Loss: 0.0230 Train Accuracy: 0.9931, Validation Accuracy: 0.9936in 31.9s
Epoch 7/10: Train Loss: 0.0203 Train Accuracy: 0.9935, Validation Accuracy: 0.9934in 33.3s
Epoch 8/10: Train Loss: 0.0192 Train Accuracy: 0.9939, Validation Accuracy: 0.9920in 32.1s
Epoch 9/10: Train Loss: 0.0165 Train Accuracy: 0.9951, Validation Accuracy: 0.9934in 29.1s
Epoch 10/10: Train Loss: 0.0132 Train Accuracy: 0.9957, Validation Accuracy: 0.9936in 32.4s


### SE-Inception module (SE block with inception module)

In [68]:
class SE_incep(nn.Module):
  def __init__(self,n_inputs):
    super().__init__()
    self.inception = InceptionModule(n_inputs)
    self.seblock = SEBlock(n_inputs=288)  # the inception module has 288 output channels

  def forward(self,X):
    return self.seblock(self.inception(X))

Create a simple model with:

- conv layer, maxpool
- SE-inception module, maxpool
- two fully connected layers with dropout
- output layer

Same as model2 with an SE-Inception module instead of an inception module.

In [69]:
model5 = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=64, kernel_size=7, padding='same'),
    nn.ReLU(),
    SE_incep(n_inputs=64),
    nn.MaxPool2d(kernel_size=2), # this divides the height and width by 2
    nn.Flatten(),
    nn.Linear(in_features = 56448, out_features = 144), # 288 out channels, 14x14 spacial dimension. Thus
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features = 144, out_features = 77),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features = 77, out_features = 10)
).to(device)

Train model5

In [70]:
learning_rate = 0.001
betas = (0.9, 0.999)
optimizer = torch.optim.NAdam(model5.parameters(), betas = betas, lr = learning_rate)
xentropy = nn.CrossEntropyLoss()

In [71]:
best_model, best_acc = train(model5, optimizer, xentropy, train_loader, valid_loader, 10)

Epoch 1/10: Train Loss: 0.3298 Train Accuracy: 0.9011, Validation Accuracy: 0.9806in 27.0s
Epoch 2/10: Train Loss: 0.1371 Train Accuracy: 0.9626, Validation Accuracy: 0.9850in 26.5s
Epoch 3/10: Train Loss: 0.1023 Train Accuracy: 0.9721, Validation Accuracy: 0.9880in 26.9s
Epoch 4/10: Train Loss: 0.0875 Train Accuracy: 0.9763, Validation Accuracy: 0.9830in 27.0s
Epoch 5/10: Train Loss: 0.0739 Train Accuracy: 0.9793, Validation Accuracy: 0.9888in 26.6s
Epoch 6/10: Train Loss: 0.0646 Train Accuracy: 0.9816, Validation Accuracy: 0.9904in 26.6s
Epoch 7/10: Train Loss: 0.0558 Train Accuracy: 0.9838, Validation Accuracy: 0.9896in 26.5s
Epoch 8/10: Train Loss: 0.0569 Train Accuracy: 0.9839, Validation Accuracy: 0.9884in 26.9s
Epoch 9/10: Train Loss: 0.0475 Train Accuracy: 0.9865, Validation Accuracy: 0.9912in 26.5s
Epoch 10/10: Train Loss: 0.0440 Train Accuracy: 0.9871, Validation Accuracy: 0.9902in 26.5s
